# Test 02 — 2D3V 网格与粒子推进

检查 `PeriodicGrid` 的每个函数。左图展示 x-y 网格节点和周期包装后的位置；右侧检查速度有 z 分量时粒子仍留在固定 z 平面。

In [ ]:
import os, sys
sys.path.insert(0, os.environ.get('BORIS_MODULE_DIR', '../build'))
import matplotlib.pyplot as plt
import boris_cpp as boris
%matplotlib inline

origin = boris.Vec3(1, 2, 7)
extent = boris.Vec3(4, 6, 0)
grid = boris.PeriodicGrid(origin, extent, (4, 3, 1))
assert tuple(grid.cells) == (4, 3, 1)
assert grid.size() == 12
dx = grid.spacing()
assert (dx.x, dx.y, dx.z) == (1.0, 2.0, 0.0)
assert grid.wrap_index(-1, 4) == 3
assert grid.wrap_index(4, 4) == 0
assert grid.index(-1, 3, 8) == 9
wrapped = grid.wrap_position(boris.Vec3(-1, 9, 123))
assert (wrapped.x, wrapped.y, wrapped.z) == (3, 3, 123)

fig, ax = plt.subplots(figsize=(6, 5))
for i in range(grid.cells[0]):
    for j in range(grid.cells[1]):
        ax.scatter(origin.x + i * dx.x, origin.y + j * dx.y, color='steelblue')
        ax.text(origin.x + i * dx.x + 0.05, origin.y + j * dx.y + 0.05, f'({i},{j})', fontsize=8)
ax.add_patch(plt.Rectangle((origin.x, origin.y), extent.x, extent.y, fill=False, color='black'))
ax.scatter(-1, 9, marker='x', s=100, color='red', label='before wrapping')
ax.scatter(wrapped.x, wrapped.y, marker='*', s=150, color='darkorange', label='after wrapping')
ax.set(xlabel='x', ylabel='y', title='2D PeriodicGrid nodes', xlim=(-2, 6), ylim=(0, 11), aspect='equal')
ax.grid(True); ax.legend(); plt.show()

# z 方向电场改变 vz，但不改变粒子的 z 坐标。
mesh = boris.UniformMesh(boris.Vec3(0,0,0), boris.Vec3(4,4,0), (4,4,1), boris.Vec3(0,0,1), boris.Vec3(0,0,0))
particle = boris.Particle(boris.Vec3(1,1,9), boris.Vec3(0,0,0), 1, 1)
boris.push(particle, 0.1, mesh)
assert abs(particle.velocity.z - 0.1) < 1e-12
assert particle.position.z == 9
print('网格函数与 2D3V 粒子行为均通过。')